# 01 — Extraction, audit et anonymisation

Le cadrage détaillé et les décisions assumées sont documentés séparément dans `reports/cadrage_jour1.md` — ce notebook exécute le pipeline correspondant et vérifie chaque étape.

**⚠️ Le fichier source `data/raw/Localites_raw.xlsx` contient des données personnelles réelles (emails, GPS précis). Il ne doit jamais être committé ni republié.**

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import anonymize
import features

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH = '../data/raw/Localites_raw.xlsx'

## 1. Chargement et audit des 29 feuilles

On charge l'ensemble du classeur en mémoire (29 feuilles) et on vérifie immédiatement le nombre de feuilles, cohérent avec ce qu'annonçait le plan de mise en œuvre.

In [4]:
# Ouvre le classeur Excel "en survol" (sans encore charger les données),
# ce qui permet de lister les feuilles sans tout lire d'un coup
xls = pd.ExcelFile(RAW_PATH, engine='openpyxl')

# Affiche combien de feuilles contient le classeur (on attend 29)
print(f"{len(xls.sheet_names)} feuilles trouvées")

# Pour chaque nom de feuille, lit réellement son contenu et le range dans un dictionnaire
# -> sheets['Localités'] donnera par exemple le tableau (DataFrame) de la feuille "Localités"
sheets = {name: pd.read_excel(xls, sheet_name=name) for name in xls.sheet_names}

# Parcourt ce dictionnaire feuille par feuille...
for name, df in sheets.items():
    # ...et affiche son nom + ses dimensions (nombre de lignes, nombre de colonnes)
    print(f"{name:30s} shape={df.shape}")

29 feuilles trouvées
Projets_Config                 shape=(1, 11)
Geo_Config                     shape=(9, 5)
Localités                      shape=(55, 9)
Categories_Taches              shape=(3, 4)
Type_Travaux                   shape=(6, 5)
Parametres_Poids               shape=(13, 7)
Referentiels_Snapshot          shape=(110, 2)
Catalogue_Materiel             shape=(28, 8)
Objectifs                      shape=(1003, 10)
Interventions                  shape=(637, 11)
Details_Intervention           shape=(1584, 15)
Journal_Chantier               shape=(36, 15)
Photos_Chantier                shape=(31, 4)
Utilisateurs                   shape=(7, 7)
Utilisateurs_Projets           shape=(7, 3)
Roles                          shape=(7, 4)
Filtre_Rapport                 shape=(1, 3)
Filtre_Performance             shape=(7, 9)
Helper_Calcul                  shape=(997, 15)
Anomalies_Taches               shape=(6, 7)
Top_Localites                  shape=(55, 10)
Top_Localites_Tri             

Projets_Config                 shape=(1, 11)
Geo_Config                     shape=(9, 5)
Localités                      shape=(55, 9)
Categories_Taches              shape=(3, 4)
Type_Travaux                   shape=(6, 5)
Parametres_Poids               shape=(13, 7)
Referentiels_Snapshot          shape=(110, 2)
Catalogue_Materiel             shape=(28, 8)
Objectifs                      shape=(1003, 10)
Interventions                  shape=(637, 11)
Details_Intervention           shape=(1584, 15)
Journal_Chantier               shape=(36, 15)
Photos_Chantier                shape=(31, 4)
Utilisateurs                   shape=(7, 7)
Utilisateurs_Projets           shape=(7, 3)
Roles                          shape=(7, 4)
Filtre_Rapport                 shape=(1, 3)
Filtre_Performance             shape=(7, 9)
Helper_Calcul                  shape=(997, 15)
Anomalies_Taches               shape=(6, 7)
Top_Localites                  shape=(55, 10)
Top_Localites_Tri              shape=(55, 10)
Histo

### 1.1 Points de vigilance détectés à l'audit

Le détail complet est dans `reports/audit_feuilles_localites.md`. Les trois points qui influencent directement le code ci-dessous :

1. **Échelle incohérente** : `Top_Localites` exprime l'avancement en fraction (0–1), `Historique_Avnt_Geo` en pourcentage (0–100) — standardisé en pourcentage dans `src/features.py`.
2. **Lignes entièrement vides** (artefacts d'export) dans `Objectifs` (6), `Details_Intervention` (3), `Interventions` (1) — supprimées avant tout traitement.
3. **Deux modes de saisie terrain** dans `Details_Intervention` (`➕ Saisie du jour` / `🎯 Cumul total à date`) — la colonne déjà calculée `Quantite_Nette_Calculee` les réconcilie, utilisée telle quelle.

In [5]:
# Aperçu du déséquilibre de classe pour la future cible "retard" (Modèle C, Jour 3)
print(sheets['Top_Localites']['Statut'].value_counts())
print()
print("Confirme : classes fortement déséquilibrées, accuracy à proscrire pour l'évaluation future.")

Statut
En Cours       52
Terminé         2
Non Démarré     1
Name: count, dtype: int64

Confirme : classes fortement déséquilibrées, accuracy à proscrire pour l'évaluation future.


## 2. Anonymisation

Périmètre PII identifié à l'audit : emails + noms (`Utilisateurs`, `Roles`, `Utilisateurs_Projets`, `Filtre_Performance`, `Interventions`, `Journal_Chantier`), téléphone (`Utilisateurs`), GPS précis (`Localités`, `Interventions`, `Journal_Chantier`).

Un seul mapping `email -> agent_id` est construit (7 personnes, réutilisées à l'identique dans 6 feuilles) puis appliqué de façon cohérente partout.

In [6]:
# Construit un dictionnaire de correspondance {email réel : identifiant anonyme}
# en parcourant toutes les feuilles du classeur (probablement en repérant les
# colonnes contenant des emails et en générant un ID stable pour chaque email unique)
mapping = anonymize.build_email_mapping(sheets)

print("Mapping construit :")

# Parcourt chaque paire (email d'origine, identifiant anonymisé) du dictionnaire
for email, agent_id in mapping.items():
    # Affiche la correspondance sous forme "AGENT_ID  <-  email"
    # pour vérifier visuellement que le mapping est cohérent avant de l'appliquer
    print(f"  {agent_id}  <-  {email}")

Mapping construit :
  agent_01  <-  dmedcos@gmail.com
  agent_02  <-  jolissaint.doudou@aiec-ci.com
  agent_03  <-  ludovis1989@gmail.com
  agent_04  <-  joanzanm@gmail.com
  agent_05  <-  gnansounoularios@gmail.com
  agent_06  <-  mafoti16@gmail.com
  agent_07  <-  mental.en.action@gmail.com


In [16]:
# Analyse l'ensemble des feuilles (sheets) à la recherche de données à caractère
# personnel (PII : emails, numéros de téléphone, coordonnées GPS précises, noms, etc.)
pii_report = anonymize.audit_pii(sheets)
pii_report

,Feuille,Colonne,Type_PII,Valeurs_non_nulles,Traitement
0,Filtre_Performance,Email_Utilisateur,email,7,email -> agent_id
1,Filtre_Performance,Nom_Complet,nom,7,colonne supprimée
2,Interventions,Coordonnees_GPS,GPS,635,colonne supprimée
3,Interventions,Utilisateur,email,636,email -> agent_id
4,Journal_Chantier,Localisation,GPS,29,colonne supprimée
5,Journal_Chantier,Utilisateur,email,36,email -> agent_id
6,Localités,Coordonnees_GPS,GPS,55,colonne supprimée
7,Roles,Email,email,7,email -> agent_id
8,Roles,Nom_Complet,nom,7,colonne supprimée
9,Utilisateurs,Email,email,7,email -> agent_id


In [8]:
anon_sheets = anonymize.anonymize_sheets(sheets, mapping)

# Nettoyage des lignes totalement vides identifiées à l'audit
for name in list(anon_sheets.keys()):
    before = len(anon_sheets[name])
    anon_sheets[name] = anon_sheets[name].dropna(how='all').reset_index(drop=True)
    after = len(anon_sheets[name])
    if before != after:
        print(f"{name}: {before - after} ligne(s) vide(s) supprimée(s)")

print()
print("Colonnes Interventions après anonymisation :", list(anon_sheets['Interventions'].columns))
print("Colonnes Localités après anonymisation     :", list(anon_sheets['Localités'].columns))

Objectifs: 6 ligne(s) vide(s) supprimée(s)
Interventions: 1 ligne(s) vide(s) supprimée(s)
Details_Intervention: 3 ligne(s) vide(s) supprimée(s)

Colonnes Interventions après anonymisation : ['ID_Intervention', 'Date_Saisie', 'ID_Projet', 'ID_Geo', 'Filtre_Geo_1', 'Filtre_Geo_2', 'ID_Localite', 'Categorie', 'Type_Travaux', 'agent_id']
Colonnes Localités après anonymisation     : ['ID_Localite', 'ID_Projet', 'ID_Geo', 'Numero', 'Nom_Geo_1', 'Nom_Geo_2', 'Localite', 'Actif']


In [9]:
# Vérification de sécurité avant tout export : aucun email/GPS/téléphone résiduel
import re

def contient_pii(df):
    texte = df.astype(str).apply(lambda col: col.str.cat(sep=' '), axis=0).str.cat(sep=' ')
    email = bool(re.search(r'@gmail\.com|@aiec-ci\.com', texte))
    gps = bool(re.search(r'\d\.\d{5,}, ?\d\.\d{5,}', texte))
    return email or gps

alertes = [name for name, df in anon_sheets.items() if contient_pii(df)]
assert not alertes, f"PII résiduelle détectée dans : {alertes}"
print("OK : aucune PII résiduelle détectée sur les 29 feuilles anonymisées.")

OK : aucune PII résiduelle détectée sur les 29 feuilles anonymisées.


In [10]:
import os
os.makedirs('../data/processed/sheets_anonymises', exist_ok=True)
for name, df in anon_sheets.items():
    df.to_csv(f'../data/processed/sheets_anonymises/{name}.csv', index=False)
print(f"{len(anon_sheets)} feuilles anonymisées exportées vers data/processed/sheets_anonymises/")

29 feuilles anonymisées exportées vers data/processed/sheets_anonymises/


## 3. Construction des tables pivot (section 4 du plan)

**Décision de cadrage (voir `reports/cadrage_jour1.md`, section 4, D2) :** le grain unique Localité × Tâche × Date visé initialement se heurte à deux grains natifs distincts dans les données (Localité × Tâche × Matériel d'un côté, Département × Semaine de l'autre). Plutôt que de forcer une fusion artificielle, trois tables complémentaires sont produites :

| Table | Grain | Modèle(s) alimenté(s) |
|---|---|---|
| A — `table_pivot_anonymisee.csv` | Localité × Tâche × Matériel | A (ressources) |
| B — `table_pivot_ressources_temporelle_anonymisee.csv` | Localité × Tâche × Date | A (évolution), C (risque) |
| C — `table_avancement_departement_semaine_anonymisee.csv` | Département × Semaine | B (avancement), C (risque) |

Ces trois tables sont construites à partir des données **originales** (non anonymisées) car elles ne mobilisent aucune colonne PII — elles sont donc « anonymisées » par construction, sans perte d'information utile au modèle.

In [11]:
t_ressources = features.build_table_ressources_detail(sheets)
print("Table A - ressources détail :", t_ressources.shape)
t_ressources.head()

Table A - ressources détail : (997, 23)


,ID_Helper,ID_Objectif,ID_Localite,Localite,Departement,Commune,ID_Tache,Categorie,Designation,ID_Materiel,Unite,Qte_Prevue,Qte_Realisee,Quantite_Restante,Taux_Realisation,Poids,Qte_Prevue_Total,Avancement_Pondere,Statut,Avancement_HTA_Localite_pct,Avancement_BT_Localite_pct,Avancement_Global_Localite_pct,Nb_Anomalies_Signalees
0,HC-342,e660d565,ALI_Kan_02,Albarika,ALIBORI,Kandi,P2AE_Cable_BT,Travaux BT,PRC 50mm²,8e6b4946,m,7505.0,6339.90,1165.10,0.844757,0.3111,11615.0,0.169810,En Cours,56.088723,76.977296,67.828101,0
1,HC-343,dea88019,ALI_Kan_02,Albarika,ALIBORI,Kandi,P2AE_Cable_BT,Travaux BT,PRC 70mm²,59dc5a38,m,4110.0,2407.07,1702.93,0.585662,0.3111,11615.0,0.064472,En Cours,56.088723,76.977296,67.828101,0
2,HC-341,ddef61fd,ALI_Kan_02,Albarika,ALIBORI,Kandi,P2AE_Cable_HTA,Travaux HTA,ASTER 54mm²,e9e0a228,m,350.0,298.00,52.00,0.851429,0.3333,350.0,0.283781,En Cours,56.088723,76.977296,67.828101,0
3,HC-344,4db2fa91,ALI_Kan_02,Albarika,ALIBORI,Kandi,P2AE_DMT,Travaux HTA,Point Double DMT-CC,a4425df9,U,2.0,0.00,2.00,0.000000,0.0833,2.0,0.000000,En Cours,56.088723,76.977296,67.828101,0
4,HC-347,495fe90e,ALI_Kan_02,Albarika,ALIBORI,Kandi,P2AE_EP,Travaux BT,EP,fbeef696,U,128.0,53.00,75.00,0.414062,0.1556,128.0,0.064428,En Cours,56.088723,76.977296,67.828101,0


In [12]:
t_temporelle = features.build_table_ressources_temporelle(sheets)
print("Table B - ressources temporelle :", t_temporelle.shape)
t_temporelle.head()

Table B - ressources temporelle : (910, 11)


,ID_Localite,Localite,Departement,ID_Tache,Date,Jours_Ecoules_Depuis_Debut_Chantier,Quantite_Nette_Jour,Quantite_Cumulee_Realisee,Qte_Prevue_Tache,Quantite_Restante_Tache,Taux_Realisation_Tache
0,ALI_Gog_01,Wara,ALIBORI,P2AE_Cable_BT,2026-08-22,642,3792.33,3792.33,5040.0,1247.67,0.752446
1,ALI_Gog_01,Wara,ALIBORI,P2AE_Cable_HTA,2026-08-16,636,75.00,75.00,80.0,5.00,0.937500
2,ALI_Gog_01,Wara,ALIBORI,P2AE_EP,2026-08-22,642,40.00,40.00,57.0,17.00,0.701754
3,ALI_Gog_01,Wara,ALIBORI,P2AE_Fouille_BT,2026-04-08,506,59.00,59.00,117.0,58.00,0.504274
4,ALI_Gog_01,Wara,ALIBORI,P2AE_Fouille_BT,2026-04-20,518,-1.00,58.00,117.0,59.00,0.495726


In [13]:
t_avancement = features.build_table_avancement_departement_semaine(sheets)
print("Table C - avancement département x semaine :", t_avancement.shape)
t_avancement.head(10)

Table C - avancement département x semaine : (100, 10)


,Departement,Semaine,Num_Semaine,Jours_Ecoules_Depuis_Debut_Chantier,Jours_Avant_Fin_Contractuelle,Avancement_HTA_%,Avancement_BT_%,Avancement,Avancement_Theorique_Global_pct,Ecart_Avancement_Global_pct
0,ALIBORI,2026-04-13,S16-2026,511.0,159.0,5.00,30.00,20.00,76.268657,-56.268657
1,ALIBORI,2026-04-20,S17-2026,518.0,152.0,6.00,31.01,21.02,77.313433,-56.293433
2,ALIBORI,2026-04-27,S18-2026,525.0,145.0,6.01,33.01,22.01,78.358209,-56.348209
3,ALIBORI,2026-05-04,S19-2026,532.0,138.0,7.00,36.80,24.01,79.402985,-55.392985
4,ALIBORI,2026-05-11,S20-2026,539.0,131.0,9.01,36.80,25.32,80.447761,-55.127761
5,ALIBORI,2026-05-18,S21-2026,546.0,124.0,10.58,36.80,26.32,81.492537,-55.172537
6,ALIBORI,2026-05-25,S22-2026,553.0,117.0,16.24,41.70,30.55,82.537313,-51.987313
7,ALIBORI,2026-06-01,S23-2026,560.0,110.0,21.02,43.50,33.70,83.582090,-49.882090
8,ALIBORI,2026-06-08,S24-2026,567.0,103.0,26.36,43.91,36.18,84.626866,-48.446866
9,ALIBORI,2026-06-15,S25-2026,574.0,96.0,26.27,45.45,37.09,85.671642,-48.581642


### 3.1 Limite méthodologique assumée — trajectoire théorique

La trajectoire de référence (table C) est calculée entre la date de démarrage administratif du marché (18/11/2024) et la date de fin contractuelle du Gantt par département. Le suivi ElecTrack Pro ne démarre lui qu'en avril 2026. L'écart observé/théorique est donc mécaniquement très négatif en début de série — **ce n'est pas un signal de retard réel**, mais un artefact de la période d'activités préliminaires incluse dans la trajectoire. Détail complet dans `reports/cadrage_jour1.md` (décision D4). Le raffinement (ancrer la trajectoire sur le début réel des travaux physiques) est reporté au Jour 3.

In [14]:
t_avancement[t_avancement['Departement']=='ATACORA'][['Departement','Semaine','Avancement','Avancement_Theorique_Global_pct','Ecart_Avancement_Global_pct']].head(6)

,Departement,Semaine,Avancement,Avancement_Theorique_Global_pct,Ecart_Avancement_Global_pct
20,ATACORA,2026-04-13,31.00,80.472441,-49.472441
21,ATACORA,2026-04-20,33.00,81.574803,-48.574803
22,ATACORA,2026-04-27,34.02,82.677165,-48.657165
23,ATACORA,2026-05-04,37.01,83.779528,-46.769528
24,ATACORA,2026-05-11,37.34,84.881890,-47.541890
25,ATACORA,2026-05-18,37.34,85.984252,-48.644252


In [15]:
t_ressources.to_csv('../data/processed/table_pivot_anonymisee.csv', index=False)
t_temporelle.to_csv('../data/processed/table_pivot_ressources_temporelle_anonymisee.csv', index=False)
t_avancement.to_csv('../data/processed/table_avancement_departement_semaine_anonymisee.csv', index=False)

print("Tables pivot sauvegardées dans data/processed/ :")
print(" - table_pivot_anonymisee.csv                         ", t_ressources.shape)
print(" - table_pivot_ressources_temporelle_anonymisee.csv    ", t_temporelle.shape)
print(" - table_avancement_departement_semaine_anonymisee.csv ", t_avancement.shape)

Tables pivot sauvegardées dans data/processed/ :
 - table_pivot_anonymisee.csv                          (997, 23)
 - table_pivot_ressources_temporelle_anonymisee.csv     (910, 11)
 - table_avancement_departement_semaine_anonymisee.csv  (100, 10)


## 4. Bilan du Jour 1

- [x] Cadrage écrit (`reports/cadrage_jour1.md`)
- [x] Audit des 29 feuilles (`reports/audit_feuilles_localites.md`)
- [x] Extraction et anonymisation immédiate (29 feuilles exportées, PII vérifiée absente)
- [x] Table pivot construite (3 tables complémentaires, décision documentée)

**Prochaine étape (Jour 2) :** EDA ciblée sur les trois tables pivot + baselines (régression linéaire pour A, tendance simple pour B, régression logistique pour C).